In [ ]:
!ls /kaggle/input/datasets/souravmdileep1607/deepfashion-mini-project-dataset/deepfashion_dataset/data/split

In [ ]:
# ============================================
# VR MINI PROJECT – CLASSIFICATION TRAINING
# FULL RUBRIC-COMPLIANT NOTEBOOK
# ============================================

import os
import torch
import timm
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
import zipfile


# ============================================
# DEVICE CONFIGURATION
# ============================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================
# DATASET PATH
# ============================================

DATA_PATH = "/kaggle/input/datasets/souravmdileep1607/deepfashion-mini-project-dataset/deepfashion_dataset/data/split"

TRAIN_IMG = f"{DATA_PATH}/train/images"
VAL_IMG   = f"{DATA_PATH}/val/images"

TRAIN_CSV = f"{DATA_PATH}/train/labels.csv"
VAL_CSV   = f"{DATA_PATH}/val/labels.csv"


# ============================================
# DATASET CLASS
# ============================================

class FashionDataset(Dataset):

    def __init__(self, img_dir, csv_file, transform=None):

        self.img_dir = img_dir
        self.df = pd.read_csv(csv_file)
        self.transform = transform


    def __len__(self):

        return len(self.df)


    def __getitem__(self, idx):

        img_name = self.df.iloc[idx,0]

        img_path = os.path.join(self.img_dir, img_name)

        image = Image.open(img_path).convert("RGB")

        label = self.df.iloc[idx,1:].values.astype(np.float32)

        if self.transform:

            image = self.transform(image)

        return image, torch.tensor(label)


# ============================================
# TRANSFORMS
# ============================================

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])


val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])


# ============================================
# LOAD DATA
# ============================================

train_dataset = FashionDataset(
    TRAIN_IMG,
    TRAIN_CSV,
    train_transform
)

val_dataset = FashionDataset(
    VAL_IMG,
    VAL_CSV,
    val_transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)


# ============================================
# TRAIN FUNCTION
# ============================================

def train_model(model_name, pretrained):

    mode = "TL" if pretrained else "Scratch"

    print(f"\n===== Training {model_name} ({mode}) =====\n")

    model = timm.create_model(
        model_name,
        pretrained=pretrained,
        num_classes=5
    )

    model.to(device)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=1e-4
    )

    EPOCHS = 10


    # TRAIN LOOP

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0


        for images, labels in tqdm(train_loader):

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()


        avg_loss = total_loss / len(train_loader)

        print(f"{model_name} ({mode}) Epoch {epoch+1}/10 Loss:", avg_loss)


    # VALIDATION LOOP

    model.eval()

    val_loss = 0

    all_preds = []
    all_targets = []


    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            all_preds.append(torch.sigmoid(outputs).cpu().numpy())
            all_targets.append(labels.cpu().numpy())


    val_loss /= len(val_loader)

    print(f"{model_name} ({mode}) Validation Loss:", val_loss)


    # METRICS

    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    preds_binary = (all_preds > 0.5).astype(int)


    precision_macro = precision_score(
        all_targets,
        preds_binary,
        average="macro"
    )


    recall_macro = recall_score(
        all_targets,
        preds_binary,
        average="macro"
    )


    f1_macro = f1_score(
        all_targets,
        preds_binary,
        average="macro"
    )


    f1_micro = f1_score(
        all_targets,
        preds_binary,
        average="micro"
    )


    print("Precision Macro:", precision_macro)
    print("Recall Macro:", recall_macro)
    print("F1 Macro:", f1_macro)
    print("F1 Micro:", f1_micro)


    # ROC-AUC PER CLASS

    for i in range(5):

        auc = roc_auc_score(
            all_targets[:,i],
            all_preds[:,i]
        )

        print(f"AUC Class {i}:", auc)


    # SAVE MODEL

    save_path = f"/kaggle/working/{model_name}_{mode}.pth"

    torch.save(model.state_dict(), save_path)

    print("Saved:", save_path)


    return f1_macro


# ============================================
# TRAIN ALL MODELS
# ============================================

results = {}

models = [

    "resnet50",
    "efficientnet_b0",
    "mobilenetv3_small_100"

]


for m in models:

    results[f"{m}_TL"] = train_model(m, True)

    results[f"{m}_Scratch"] = train_model(m, False)


# ============================================
# FINAL RESULTS SUMMARY
# ============================================

print("\n===== FINAL RESULTS =====\n")

for k,v in results.items():

    print(k, ":", v)


# ============================================
# ZIP ALL MODELS FOR SAFE DOWNLOAD
# ============================================

zip_path = "/kaggle/working/classification_models_10epochs.zip"

with zipfile.ZipFile(zip_path, 'w') as z:

    for file in os.listdir("/kaggle/working"):

        if file.endswith(".pth"):

            z.write(f"/kaggle/working/{file}", file)


print("\nZIP CREATED:", zip_path)

print("\nTRAINING COMPLETE — ALL 6 MODELS FINISHED SUCCESSFULLY")

In [ ]:
import os
import zipfile

zip_path = "/kaggle/working/saved_models_10epochs.zip"

with zipfile.ZipFile(zip_path, 'w') as z:
    for file in os.listdir("/kaggle/working"):
        if file.endswith(".pth"):
            z.write(f"/kaggle/working/{file}", file)

print("ZIP created at:", zip_path)

In [7]:
!ls /kaggle/input/datasets/souravmdileep1607/deepfashion-mini-project-dataset/deepfashion_dataset/data/split

test  train  val


In [2]:
import shutil

shutil.copy(
    "/kaggle/input/datasets/souravmdileep1607/classification-models-10epochs-zip/resnet50_TL.pth",
    "/kaggle/working/resnet50_TL.pth"
)

print("Checkpoint copied successfully.")

Checkpoint copied successfully.


In [3]:
!ls /kaggle/working


resnet50_TL.pth


In [4]:
# ============================================
# CONTINUE TRAINING RESNET50 TL (10 → 15 epochs)
# ============================================

import torch
import timm
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm

import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader


# =============================
# DEVICE
# =============================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)


# =============================
# DATASET PATH
# =============================

DATA_PATH = "/kaggle/input/datasets/souravmdileep1607/deepfashion-mini-project-dataset/deepfashion_dataset/data/split"


# =============================
# DATASET CLASS
# =============================

class FashionDataset(Dataset):

    def __init__(self, img_dir, csv_file, transform=None):

        self.img_dir = img_dir
        self.df = pd.read_csv(csv_file)
        self.transform = transform


    def __len__(self):

        return len(self.df)


    def __getitem__(self, idx):

        img_name = self.df.iloc[idx,0]

        img_path = f"{self.img_dir}/{img_name}"

        image = Image.open(img_path).convert("RGB")

        label = self.df.iloc[idx,1:].values.astype(np.float32)

        if self.transform:

            image = self.transform(image)

        return image, torch.tensor(label)


# =============================
# TRANSFORMS
# =============================

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])


# =============================
# DATA LOADERS
# =============================

train_dataset = FashionDataset(
    f"{DATA_PATH}/train/images",
    f"{DATA_PATH}/train/labels.csv",
    train_transform
)

val_dataset = FashionDataset(
    f"{DATA_PATH}/val/images",
    f"{DATA_PATH}/val/labels.csv",
    val_transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)


# =============================
# LOAD MODEL FROM 10-EPOCH CHECKPOINT
# =============================

model = timm.create_model(
    "resnet50",
    pretrained=True,
    num_classes=5
)

model.load_state_dict(
    torch.load("/kaggle/working/resnet50_TL.pth")
)

model.to(device)


# =============================
# LOSS + OPTIMIZER
# =============================

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-5   # smaller LR for continuation
)


# =============================
# CONTINUE TRAINING (5 MORE EPOCHS)
# =============================

print("\nContinuing training: Epoch 11 → 15\n")

for epoch in range(5):

    model.train()

    total_loss = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+11}/15 Loss:", total_loss/len(train_loader))


# =============================
# VALIDATION AGAIN
# =============================

model.eval()

val_loss = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        val_loss += loss.item()

val_loss /= len(val_loader)

print("\nValidation Loss after 15 epochs:", val_loss)


# =============================
# SAVE IMPROVED MODEL
# =============================

torch.save(
    model.state_dict(),
    "/kaggle/working/resnet50_TL_15epochs.pth"
)

print("\nSaved improved model:")
print("/kaggle/working/resnet50_TL_15epochs.pth")

Using device: cuda



Continuing training: Epoch 11 → 15



100%|██████████| 438/438 [05:57<00:00,  1.22it/s]


Epoch 11/15 Loss: 0.0516948847835841


100%|██████████| 438/438 [04:43<00:00,  1.54it/s]


...

Epoch 13/15 Loss: 0.04385728632912176


100%|██████████| 438/438 [04:44<00:00,  1.54it/s]


Epoch 14/15 Loss: 0.042577770929868634


100%|██████████| 438/438 [04:39<00:00,  1.57it/s]


Epoch 15/15 Loss: 0.04082991813466837

Validation Loss after 15 epochs: 0.39920299912386753

Saved improved model:
/kaggle/working/resnet50_TL_15epochs.pth


In [8]:
# ============================================
# STEP 1 — INSTALL YOLOv8
# ============================================

!pip install ultralytics -q


# ============================================
# STEP 2 — IMPORT LIBRARIES
# ============================================

from ultralytics import YOLO
import os
import yaml


# ============================================
# STEP 3 — DEFINE DATASET PATH
# ============================================

DATA_ROOT = "/kaggle/input/datasets/souravmdileep1607/deepfashion-mini-project-dataset/deepfashion_dataset/data/split"


train_images = f"{DATA_ROOT}/train/images"
val_images   = f"{DATA_ROOT}/val/images"


# ============================================
# STEP 4 — CREATE dataset.yaml FILE
# ============================================

dataset_yaml = {

    "path": DATA_ROOT,

    "train": "train/images",

    "val": "val/images",

    "names": {

        0: "short sleeve top",
        1: "trousers",
        2: "shorts",
        3: "long sleeve top",
        4: "skirt"

    }

}


yaml_path = "/kaggle/working/deepfashion_detection.yaml"


with open(yaml_path, "w") as f:

    yaml.dump(dataset_yaml, f)


print("dataset.yaml created successfully")


# ============================================
# STEP 5 — LOAD YOLOv8 MODEL
# ============================================

model = YOLO("yolov8n.pt")


# ============================================
# STEP 6 — TRAIN MODEL
# ============================================

results = model.train(

    data=yaml_path,

    epochs=20,

    imgsz=640,

    batch=16,

    device=0,

    project="/kaggle/working",

    name="deepfashion_detection"

)


print("Training completed")


# ============================================
# STEP 7 — VALIDATION METRICS
# ============================================

metrics = model.val()

print("\nDetection Metrics:")
print(metrics)


# ============================================
# STEP 8 — LOCATE BEST WEIGHTS
# ============================================

best_model_path = "/kaggle/working/deepfashion_detection/weights/best.pt"

print("\nBest detection model saved at:")
print(best_model_path)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.8 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
dataset.yaml created successfully
Ultralytics 8.4.27 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/deepfashion_detection.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, f

In [12]:
!ls /kaggle/working/deepfashion_detection/weights

best.pt  last.pt


In [16]:
!du -h --max-depth=1 /kaggle/working

20K	/kaggle/working/.virtual_documents
14M	/kaggle/working/deepfashion_detection
3.7M	/kaggle/working/runs
20G	/kaggle/working


In [18]:
!rm -rf /kaggle/working/full_project_backup.zip

In [19]:
!df -h

Filesystem                                                              Size  Used Avail Use% Mounted on
overlay                                                                 7.9T  6.7T  1.2T  85% /
tmpfs                                                                    64M     0   64M   0% /dev
shm                                                                      14G  116K   14G   1% /dev/shm
/dev/sda1                                                               122G   24G   99G  20% /opt/bin
/dev/loop1                                                               20G  209M   20G   2% /kaggle/lib
192.168.5.2:/data/kagglesdsdata/datasets/9825407/15360851/dhcfu1acwilp  100T   86T   15T  86% /kaggle/input/datasets/souravmdileep1607/classification-models-10epochs-zip
192.168.5.2:/data/kagglesdsdata/datasets/9740170/15224071/dhcfu1a6yu9z  100T   86T   15T  86% /kaggle/input/datasets/souravmdileep1607/deepfashion-mini-project-dataset
/dev/mapper/snap                                  